In [10]:
import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras import models

from tensorflow.keras.optimizers import Adam

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

In [11]:
IMG_HEIGHT = 128
IMG_WIDTH = 128

BATCH_SIZE = 32
SEED = 42

In [12]:
dataset_path = "/workspaces/plant_disease_detection/data/raw"

In [16]:
train_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE
)



Found 20638 files belonging to 15 classes.
Using 16511 files for training.
Found 20638 files belonging to 15 classes.
Using 4127 files for validation.


In [17]:
class_names = train_dataset.class_names

NUM_CLASSES = len(class_names)

print("Number of classes:", NUM_CLASSES)

Number of classes: 15


In [18]:
train_dataset = train_dataset.take(100)
validation_dataset = validation_dataset.take(20)

In [19]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(
    buffer_size=AUTOTUNE
)

validation_dataset = validation_dataset.prefetch(
    buffer_size=AUTOTUNE
)

In [20]:
data_augmentation = tf.keras.Sequential([

    layers.RandomFlip("horizontal"),

    layers.RandomRotation(0.2),

    layers.RandomZoom(0.2),

    layers.RandomContrast(0.2)

])

In [21]:
def build_cnn_model():

    model = models.Sequential([

        layers.Input(shape=(IMG_HEIGHT,
                            IMG_WIDTH,
                            3)),

        data_augmentation,

        layers.Rescaling(1./255),

        layers.Conv2D(32, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),

        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),

        layers.Conv2D(128, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),

        layers.Flatten(),

        layers.Dense(128, activation='relu'),

        layers.Dropout(0.3),

        layers.Dense(NUM_CLASSES,
                     activation='softmax')

    ])

    return model

In [22]:
model = build_cnn_model()

In [23]:
model.compile(

    optimizer=Adam(
        learning_rate=0.001
    ),

    loss='sparse_categorical_crossentropy',

    metrics=['accuracy']

)

In [24]:
early_stopping = EarlyStopping(

    monitor='val_loss',

    patience=5,

    restore_best_weights=True
)

checkpoint = ModelCheckpoint(

    filepath='../artifacts/best_model.keras',

    monitor='val_accuracy',

    save_best_only=True,

    verbose=1
)

reduce_lr = ReduceLROnPlateau(

    monitor='val_loss',

    factor=0.2,

    patience=3,

    verbose=1
)

In [26]:
EPOCHS = 5

history = model.fit(

    train_dataset,

    validation_data=validation_dataset,

    epochs=EPOCHS,

    callbacks=[

        early_stopping,

        checkpoint,

        reduce_lr
    ]
)

Epoch 1/5


W0000 00:00:1782003366.860673  162953 cpu_allocator_impl.cc:82] Allocation of 65028096 exceeds 10% of free system memory.
W0000 00:00:1782003367.146776  162953 cpu_allocator_impl.cc:82] Allocation of 16257024 exceeds 10% of free system memory.
W0000 00:00:1782003367.163255  162953 cpu_allocator_impl.cc:82] Allocation of 30482432 exceeds 10% of free system memory.
W0000 00:00:1782003367.451697  162953 cpu_allocator_impl.cc:82] Allocation of 23482368 exceeds 10% of free system memory.
W0000 00:00:1782003367.584609  162952 cpu_allocator_impl.cc:82] Allocation of 30482432 exceeds 10% of free system memory.


100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 718ms/step - accuracy: 0.1581 - loss: 2.5600
Epoch 1: val_accuracy improved from None to 0.40000, saving model to ../artifacts/best_model.keras

Epoch 1: finished saving model to ../artifacts/best_model.keras
100/100 ━━━━━━━━━━━━━━━━━━━━ 79s 767ms/step - accuracy: 0.2237 - loss: 2.3694 - val_accuracy: 0.4000 - val_loss: 1.9888 - learning_rate: 0.0010
Epoch 2/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 756ms/step - accuracy: 0.3849 - loss: 1.9336
Epoch 2: val_accuracy improved from 0.40000 to 0.50156, saving model to ../artifacts/best_model.keras

Epoch 2: finished saving model to ../artifacts/best_model.keras
100/100 ━━━━━━━━━━━━━━━━━━━━ 85s 803ms/step - accuracy: 0.4153 - loss: 1.8306 - val_accuracy: 0.5016 - val_loss: 1.5785 - learning_rate: 0.0010
Epoch 3/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 705ms/step - accuracy: 0.5077 - loss: 1.5883
Epoch 3: val_accuracy improved from 0.50156 to 0.62656, saving model to ../artifacts/best_model.keras

Epoch 3: finished saving mo

In [27]:
import pickle

with open('../artifacts/history.pkl', 'wb') as file:
    pickle.dump(history.history, file)